<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Buscar las Llaves: Cuadrícula, Azar, Escala Mágica y Zoom 🔑
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/07%20-%20Seleccion%20de%20Modelos%20y%20Optimizacion/Para%20Dummies/02_Random_Grid_Search_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 02 (Random Search y GridSearch)](../02_Random_Search_y_GridSearch.ipynb). Un modelo tiene «perillas» (los **hiperparámetros**) que tú debes ajustar antes de entrenarlo. Probar **todas** las combinaciones es carísimo. ¿Hay una forma más lista de buscar con pocos intentos? Aquí lo verás con analogías cotidianas y simulaciones pequeñas, **sin fórmulas**: solo *NumPy* y gráficas.

Al terminar podrás explicar:
1. Por qué, con los mismos intentos, **probar al azar** puede explorar **más** que una **cuadrícula**.
2. Qué es la **escala mágica** (logarítmica) y por qué no se prueba «de 1 en 1» cuando los valores van de 0.001 a 1000.
3. Cómo **hacer zoom**: una búsqueda amplia y luego otra en la zona prometedora.
4. La idea de las **pruebas cortas para todos y largas para los mejores** (*casting de talentos*).

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../02_Random_Search_y_GridSearch.ipynb). Antes: [Los exámenes de práctica justos (Dummies)](01_Validacion_Cruzada_Avanzada_Dummies.ipynb).

---
## 1. El radio con dos perillas 📻

Un radio viejo tiene **dos perillas**: el **dial** (que decide qué emisora oyes) y el **tono de graves** (que casi no cambia lo claro que se oye). Tienes **9 intentos** para dar con la emisora. Dos estrategias:

- **Cuadrícula 3×3**: pruebas 3 posiciones del dial **× 3** del tono. ¡Pero solo oyes **3** posiciones distintas del dial! Los otros 6 intentos repiten posiciones del dial que ya probaste (el tono casi no importa).
- **Al azar**: cada intento pone **ambas** perillas en posiciones nuevas: ¡**9** posiciones distintas del dial!

La emisora está en una posición del dial desconocida y la señal se oye bien **solo muy cerca** de ella. Repetimos el juego miles de veces con emisoras en distintos lugares:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
def senal(dial, tono, emisora):
    """Qué tan bien se oye: muy sensible al dial (campana estrecha) y casi nada al tono."""
    return 0.95 * np.exp(-((dial - emisora) ** 2) / (2 * 0.06 ** 2)) + 0.05 * np.exp(-((tono - 0.5) ** 2) / (2 * 0.3 ** 2))

juegos = 5000
emisoras = rng.uniform(0.15, 0.85, juegos)                          # dónde está la emisora en cada juego
centros = (np.arange(3) + 0.5) / 3                                  # cuadrícula 3x3: 3 posiciones por perilla
dial_c, tono_c = np.meshgrid(centros, centros); dial_c, tono_c = dial_c.ravel(), tono_c.ravel()
mejor_cuadricula = senal(dial_c[None, :], tono_c[None, :], emisoras[:, None]).max(axis=1)
dial_a, tono_a = rng.uniform(size=(juegos, 9)), rng.uniform(size=(juegos, 9))      # 9 intentos al azar
mejor_azar = senal(dial_a, tono_a, emisoras[:, None]).max(axis=1)

print(f"Mejor señal encontrada con 9 intentos -> cuadrícula 3x3: {mejor_cuadricula.mean():.2f} | al azar: {mejor_azar.mean():.2f}  (máximo posible = 1.00)")
print(f"Posiciones DISTINTAS del dial probadas: cuadrícula = 3 | al azar = 9")
print(f"El azar gana en el {np.mean(mejor_azar > mejor_cuadricula):.0%} de los juegos")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
axes[0].scatter(dial_c, tono_c, s=70, color="#dc2626"); axes[0].set_title("Cuadrícula 3×3: solo 3 posiciones del dial", fontweight="bold", fontsize=10)
axes[1].scatter(dial_a[0], tono_a[0], s=70, color="#16a34a"); axes[1].set_title("9 intentos al azar: 9 posiciones del dial", fontweight="bold", fontsize=10)
for ax in axes: ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_xlabel("Dial (la perilla que SÍ importa)"); ax.set_ylabel("Tono (casi no importa)")
plt.tight_layout(); plt.show()
assert mejor_azar.mean() > mejor_cuadricula.mean()

---
**Lección:** si **pocas perillas importan**, la cuadrícula desperdicia intentos repitiendo lo que sí importa. El azar prueba **una posición nueva de cada perilla en cada intento**. Con modelos reales casi siempre ocurre que **unos pocos hiperparámetros pesan mucho y los demás poco**, y no sabes de antemano cuáles. (**Ojo honesto:** con solo **1 o 2** perillas y un terreno suave, una cuadrícula bien puesta funciona igual de bien; la ventaja del azar aparece cuando hay **muchas** perillas.)

## 2. La escala mágica: de 1 en 1 o de 10 en 10 🔭

Algunas perillas funcionan como **zoom multiplicador**: pasar de 0.001 a 0.01 cambia **tanto** como pasar de 100 a 1000. Imagina que buscas el **precio** de una casa entre 1 millón y 1000 millones. Si pruebas **al azar en línea recta** (todos los valores igual de probables), casi **todos** tus intentos caen entre 100 y 1000 millones: ¡nunca pruebas casas baratas! La **escala mágica** (logarítmica) da la **misma oportunidad a cada «orden de magnitud»**: 1-10, 10-100, 100-1000.

In [ ]:
rng = np.random.default_rng(2)
M = 20000
en_linea_recta = rng.uniform(1, 1000, M)                              # todos los precios entre 1 y 1000 igual de probables
en_escala_magica = 10 ** rng.uniform(0, 3, M)                         # cada 'orden de magnitud' igual de probable

def por_orden(v):
    return [np.mean((v >= a) & (v < b)) for a, b in [(1, 10), (10, 100), (100, 1000)]]
etiquetas = ["1 a 10", "10 a 100", "100 a 1000"]
print(f"{'':>22}" + "".join(f"{e:>14}" for e in etiquetas))
print(f"{'En línea recta':>22}" + "".join(f"{p:>14.1%}" for p in por_orden(en_linea_recta)))
print(f"{'En escala mágica':>22}" + "".join(f"{p:>14.1%}" for p in por_orden(en_escala_magica)))

fig, ax = plt.subplots(figsize=(8.5, 4))
x = np.arange(3); w = 0.35
ax.bar(x - w / 2, por_orden(en_linea_recta), w, color="#dc2626", label="En línea recta"); ax.bar(x + w / 2, por_orden(en_escala_magica), w, color="#16a34a", label="En escala mágica")
ax.set_xticks(x); ax.set_xticklabels(etiquetas); ax.set_ylabel("Fracción de los intentos"); ax.legend()
ax.set_title("¿Dónde caen tus intentos? La escala mágica explora todos los órdenes de magnitud", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert por_orden(en_linea_recta)[2] > 0.89 and abs(por_orden(en_escala_magica)[0] - 1 / 3) < 0.02

---
**Lección:** en línea recta, el **90 %** de los intentos cae en la última «década»; con la escala mágica, **un tercio en cada una**. Regla: las perillas que actúan como **«multiplicadores»** (la fuerza de un castigo, qué tan «ancha» es una campana, la velocidad de aprendizaje) se buscan en **escala logarítmica**; las que son **conteos** (cuántos vecinos, qué profundidad) se buscan entre enteros.

## 3. Buscar con zoom: el mapa del tesoro 🗺️

Estrategia de los buscadores de tesoros: **primero un barrido amplio** (pocos puntos por todo el mapa) para ver **por dónde está lo bueno**; luego **un zoom** en esa zona. Es el *coarse-to-fine* (de «grueso» a «fino»). Lo ponemos a prueba contra una sola búsqueda con el **mismo número total de intentos** (24 en total: 12 + 12 frente a 24 de una vez), en **dos mapas** distintos:

- **Mapa A, «tesoro aislado»**: una colina **muy estrecha** sobre un terreno casi plano. Desde lejos **no hay pistas** de dónde está.
- **Mapa B, «colina con cima»**: una **colina ancha** (se nota desde lejos) con una **cima estrecha** en lo alto. Las primeras pruebas **dan pistas** de hacia dónde subir.

In [ ]:
def altura(x, y, cx, cy, mapa):
    d2 = (x - cx) ** 2 + (y - cy) ** 2
    if mapa == "A":                                                     # tesoro aislado: colina estrecha + ondulaciones de fondo
        return np.exp(-d2 / (2 * 0.04 ** 2)) + 0.15 * np.sin(8 * x) * np.cos(6 * y)
    return 0.7 * np.exp(-d2 / (2 * 0.22 ** 2)) + 0.3 * np.exp(-d2 / (2 * 0.04 ** 2))   # B: colina ancha con cima estrecha

rng = np.random.default_rng(3)
juegos = 3000
resumen = {}
for mapa in ("A", "B"):
    una_etapa, dos_etapas = [], []
    for _ in range(juegos):
        cx, cy = rng.uniform(0.15, 0.85, 2)
        p = rng.uniform(size=(24, 2)); una_etapa.append(altura(p[:, 0], p[:, 1], cx, cy, mapa).max())           # una sola busqueda de 24
        p1 = rng.uniform(size=(12, 2)); h1 = altura(p1[:, 0], p1[:, 1], cx, cy, mapa); top = p1[np.argsort(-h1)[:3]]   # 12 amplios
        lo, hi = np.clip(top.min(axis=0) - 0.12, 0, 1), np.clip(top.max(axis=0) + 0.12, 0, 1)                     # caja alrededor de los 3 mejores
        p2 = rng.uniform(lo, hi, size=(12, 2)); h2 = altura(p2[:, 0], p2[:, 1], cx, cy, mapa)                     # 12 con zoom
        dos_etapas.append(max(h1.max(), h2.max()))
    una_etapa, dos_etapas = np.array(una_etapa), np.array(dos_etapas)
    resumen[mapa] = (una_etapa, dos_etapas)
    print(f"Mapa {mapa}: mejor altura (máx. posible ≈ 1.0) -> una etapa de 24: {una_etapa.mean():.3f} | zoom (12 + 12): {dos_etapas.mean():.3f}  |  el zoom gana en el {np.mean(dos_etapas > una_etapa + 1e-9):.0%} de los juegos")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, mapa, tit in zip(axes, ("A", "B"), ("Mapa A: tesoro aislado (sin pistas)", "Mapa B: colina con cima (con pistas)")):
    ax.hist(resumen[mapa][0], bins=30, alpha=0.7, color="#dc2626", density=True, label="Una etapa (24)"); ax.hist(resumen[mapa][1], bins=30, alpha=0.7, color="#16a34a", density=True, label="Zoom (12 + 12)")
    ax.set_xlabel("Altura del mejor punto encontrado"); ax.set_ylabel("Densidad"); ax.legend(fontsize=8); ax.set_title(tit, fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert resumen["B"][1].mean() > resumen["B"][0].mean() + 0.01          # con pistas, el zoom ayuda
assert abs(resumen["A"][1].mean() - resumen["A"][0].mean()) < 0.03     # sin pistas, el zoom no aporta (diferencia despreciable)

---
**Lección:** el zoom **no es magia**. Funciona cuando **el barrido amplio da pistas** (mapa B: la colina ancha se nota desde lejos y se sube hasta la cima). Si el tesoro está **aislado** y no hay pistas (mapa A), el zoom **no mejora** (gana y pierde aproximadamente la mitad de las veces): es como una búsqueda más. En modelos reales, la superficie de validación suele parecerse más al **mapa B** (zonas anchas y razonables). Y si la caja del zoom es muy estrecha puedes **dejar fuera** el tesoro: haz zoom **con margen**.

## 4. El casting de talentos: pruebas cortas para todos, largas para los mejores 🎬

Hay **27 aspirantes** y probar a **uno** a fondo cuesta una hora. Solo tienes **9 horas** (¡se pueden probar a fondo solo 9!). Otra idea, la **reducción sucesiva**: una **prueba corta** (10 minutos) a los 27, te quedas con los **9 mejores**; a ellos una prueba **mediana** (30 min), te quedas con los **3 mejores**; y a esos 3 la prueba **larga** (1 hora). Cuesta **lo mismo** (9 horas), pero **miraste a 27**.

El riesgo: las pruebas cortas son **ruidosas** (un aspirante bueno puede tener un mal momento). Simulamos aspirantes con un **talento real** y pruebas cuyo ruido **baja cuanto más largas son**:

In [ ]:
rng = np.random.default_rng(4)
def prueba(talento, duracion, ruido_base=0.30):
    """Nota observada = talento + ruido; el ruido baja con la raíz de la duración de la prueba (1.0 = prueba larga)."""
    return talento + rng.normal(0, ruido_base / np.sqrt(duracion), np.shape(talento))

juegos = 4000
aciertos_casting, aciertos_pocos = 0, 0
mejor_casting, mejor_pocos = [], []
for _ in range(juegos):
    talento = rng.normal(0, 0.15, 27)                                   # el talento real de los 27 aspirantes
    mejor_real = talento.argmax()
    # Casting: 27 con prueba corta -> 9 con mediana -> 3 con larga -> 1
    vivos = np.arange(27)
    for duracion, quedan in [(1 / 6, 9), (1 / 2, 3), (1.0, 1)]:
        notas = prueba(talento[vivos], duracion); vivos = vivos[np.argsort(-notas)[:quedan]]
    elegido_casting = vivos[0]
    # Alternativa: solo 9 aspirantes (los primeros), todos con prueba larga
    notas9 = prueba(talento[:9], 1.0); elegido_pocos = notas9.argmax()
    mejor_casting.append(talento[elegido_casting]); mejor_pocos.append(talento[elegido_pocos])
    aciertos_casting += elegido_casting == mejor_real; aciertos_pocos += elegido_pocos == mejor_real
mejor_casting, mejor_pocos = np.array(mejor_casting), np.array(mejor_pocos)
print(f"Talento REAL promedio del aspirante elegido -> casting (27 → 9 → 3 → 1): {mejor_casting.mean():.3f} | solo 9 con prueba larga: {mejor_pocos.mean():.3f}")
print(f"Veces que se eligió al MEJOR de los 27 -> casting: {aciertos_casting / juegos:.0%} | solo 9 completos: {aciertos_pocos / juegos:.0%}")

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(["Casting\n(27→9→3→1)", "Solo 9 con\nprueba larga"], [mejor_casting.mean(), mejor_pocos.mean()], color=["#16a34a", "#f59e0b"])
ax.set_ylabel("Talento real promedio del elegido"); ax.set_title("Mismo presupuesto, más aspirantes mirados", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert mejor_casting.mean() > mejor_pocos.mean()

---
**Lección:** con el **mismo tiempo**, el casting elige aspirantes de **mayor talento real**, porque mira a más gente. Pero **funciona solo si la prueba corta dice algo sobre la larga**: si las pruebas cortas fueran puro ruido, descartarías a los buenos. En machine learning, la «prueba corta» es **entrenar con pocos datos o pocas vueltas** y la «larga» es entrenar a fondo (*successive halving*).

## Resumen en una frase 🎯

> **Probar todas las combinaciones es carísimo; con muchas perillas y pocas que importan, probar al azar (y en escala mágica para los «multiplicadores») explora más que una cuadrícula con el mismo número de intentos; se puede hacer zoom en la zona buena, y se pueden descartar pronto a los malos con pruebas cortas.**

### Lo que aprendiste hoy:
- ✅ **Cuadrícula vs azar**: con 9 intentos y 1 perilla que importa, la cuadrícula prueba **3** valores de esa perilla y el azar **9**.
- ✅ **Escala mágica**: en línea recta el 90 % de los intentos cae en la última «década»; en escala logarítmica, un tercio en cada una.
- ✅ **Zoom**: ayuda si la zona buena es estrecha; con zonas anchas, casi no aporta.
- ✅ **Casting**: pruebas cortas para muchos y largas para pocos mira a más candidatos por el mismo costo, si la prueba corta es informativa.

> 🎓 **Siguiente paso:** explora el [cuaderno estándar](../02_Random_Search_y_GridSearch.ipynb) (con modelos reales, `RandomizedSearchCV` desde cero y `HalvingRandomSearchCV`) y continúa con [El cazador de tesoros en la niebla (Dummies)](03_Optimizacion_Bayesiana_Dummies.ipynb).

---
##### 🛠️ Práctica 1: Una perilla de «castigo» entre 0.0001 y 10 000

En la autoevaluación de este cuaderno aparece una perilla que puede ir de **0.0001 a 10 000** (¡ocho órdenes de magnitud!). Vamos a comprobar qué pasa si la buscas en línea recta o en escala mágica. Usa `np` y un generador nuevo `rng_c = np.random.default_rng(5)`:

1. Genera 20 000 valores **en línea recta** (`en_recta`, con `rng_c.uniform(0.0001, 10000, 20000)`) y 20 000 **en escala mágica** (`en_magica = 10 ** rng_c.uniform(-4, 4, 20000)`).
2. Para cada «orden de magnitud» (de `10**-4` a `10**4`, son 8) calcula qué fracción de los intentos cae ahí, en cada método; guárdalas en `frac_recta` y `frac_magica` (listas de 8 números).
3. Comprueba con `assert` que en línea recta **la última década se lleva casi todo** (más del 85 %) y que en escala mágica cada orden recibe aproximadamente **1/8** (con margen de 0.02).

In [ ]:
# Escribe tu código aquí

# rng_c = np.random.default_rng(5)
# en_recta = rng_c.uniform(0.0001, 10000, 20000)
# en_magica = 10 ** rng_c.uniform(-4, 4, 20000)
# ordenes = [(10.0 ** e, 10.0 ** (e + 1)) for e in range(-4, 4)]      # 8 ordenes de magnitud
# frac_recta = [np.mean((en_recta >= a) & (en_recta < b)) for a, b in ordenes]
# frac_magica = [...]


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
rng_c = np.random.default_rng(5)
en_recta = rng_c.uniform(0.0001, 10000, 20000)          # todos los valores igual de probables
en_magica = 10 ** rng_c.uniform(-4, 4, 20000)           # cada orden de magnitud igual de probable

ordenes = [(10.0 ** e, 10.0 ** (e + 1)) for e in range(-4, 4)]      # 8 ordenes de magnitud: [1e-4, 1e-3), ..., [1e3, 1e4)
frac_recta = [np.mean((en_recta >= a) & (en_recta < b)) for a, b in ordenes]
frac_magica = [np.mean((en_magica >= a) & (en_magica < b)) for a, b in ordenes]

for (a, b), fr, fm in zip(ordenes, frac_recta, frac_magica):
    print(f"[{a:g}, {b:g}): linea recta = {fr:6.1%} | escala magica = {fm:6.1%}")

assert frac_recta[-1] > 0.85                                            # en linea recta casi todo cae en la ultima decada
assert np.allclose(frac_magica, 1 / 8, atol=0.02)                       # en escala magica, 1/8 por orden
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué probar 9 combinaciones **al azar** puede ser mejor que una **cuadrícula 3×3** cuando solo una perilla importa? ¿En qué situación la cuadrícula sería igual de buena?
2. Quieres probar «qué tanto castigar los errores», un número que puede ir de 0.0001 a 10 000. ¿Lo probarías «de 1 en 1» o en escala mágica? ¿Por qué?
3. Con tus palabras, ¿qué hace un *casting de talentos* y cuándo podría **fallar** (descartar al mejor)?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>